In [18]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier, PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, classification_report
from autocorrect import spell

import pandas as pd
import re

In [7]:
profanity_words = pd.read_csv('English_profanity_words.csv', nrows=1000)
profanity_words.head()

,is_offensive,text
0,0,Then go to the village pump and suggest they c...
1,1,ANTI GREEK NATIONALIS -WIKIPEDIA \n\nHi Alexik...
2,1,Dis hoe wasnt dis violent on Lottery Ticket 😂😂
3,0,It is better for Atabay not helping the banned...
4,0,"""is in CamelCase. """"SiCKO"""" is not CamelCase,..."


In [8]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

In [9]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip() 
    tokens = text.split()
    words = []
    for token in tokens:
        corrected_token = spell(token)
        stemmed_token = stemmer.stem(corrected_token)
        lemmatized_token = lemmatizer.lemmatize(stemmed_token)
        words.append(lemmatized_token)
    return " ".join(words)

In [10]:
profanity_words["processed_text"] = profanity_words['text'].apply(preprocess_text)

autocorrect.spell is deprecated,             use autocorrect.Speller instead
autocorrect.spell is deprecated,             use autocorrect.Speller instead
autocorrect.spell is deprecated,             use autocorrect.Speller instead
autocorrect.spell is deprecated,             use autocorrect.Speller instead
autocorrect.spell is deprecated,             use autocorrect.Speller instead
autocorrect.spell is deprecated,             use autocorrect.Speller instead
autocorrect.spell is deprecated,             use autocorrect.Speller instead
autocorrect.spell is deprecated,             use autocorrect.Speller instead
autocorrect.spell is deprecated,             use autocorrect.Speller instead
autocorrect.spell is deprecated,             use autocorrect.Speller instead
autocorrect.spell is deprecated,             use autocorrect.Speller instead
autocorrect.spell is deprecated,             use autocorrect.Speller instead
autocorrect.spell is deprecated,             use autocorrect.Speller instead

In [11]:
profanity_words

,is_offensive,text,processed_text
0,0,Then go to the village pump and suggest they c...,then go to the villag pump and suggest they ch...
1,1,ANTI GREEK NATIONALIS -WIKIPEDIA \n\nHi Alexik...,anti greek nation wikipedia hi alexikoua you s...
2,1,Dis hoe wasnt dis violent on Lottery Ticket 😂😂,di hoe wasnt di violent on lotteri ticket
3,0,It is better for Atabay not helping the banned...,it is better for atabay not help the ban vanda...
4,0,"""is in CamelCase. """"SiCKO"""" is not CamelCase,...",is in camelcas sick is not camelcas so the cam...
...,...,...,...
995,1,like fuccck man why yall bitches wearing wigs...,like ffcccc man whi all pitch wear win to school
996,1,I just wanna travel the world with my bitches 😩,i just wanna travel the world with my pitch
997,1,"FUCK YOU BITCH\nKiss my ass, you dickless trol...",fuck you bitch kiss my as you sick troll i hop...
998,0,"depends what bbg stands for.\n\nAlso, was the ...",depend what big stand for also wa the list for...


In [14]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(profanity_words["processed_text"])
y = profanity_words["is_offensive"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [19]:
models = {
    "Linear SVM": LinearSVC(
        class_weight="balanced"
    ),
    "SGD (Hinge Loss)": SGDClassifier(
        loss="hinge",
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ),
    "Passive Aggressive": PassiveAggressiveClassifier(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
}


In [20]:
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))


Linear SVM
Accuracy: 0.895
              precision    recall  f1-score   support

           0       0.91      0.97      0.94       165
           1       0.79      0.54      0.64        35

    accuracy                           0.90       200
   macro avg       0.85      0.76      0.79       200
weighted avg       0.89      0.90      0.89       200


SGD (Hinge Loss)
Accuracy: 0.88
              precision    recall  f1-score   support

           0       0.94      0.92      0.93       165
           1       0.64      0.71      0.68        35

    accuracy                           0.88       200
   macro avg       0.79      0.81      0.80       200
weighted avg       0.89      0.88      0.88       200


Passive Aggressive
Accuracy: 0.89
              precision    recall  f1-score   support

           0       0.93      0.94      0.93       165
           1       0.70      0.66      0.68        35

    accuracy                           0.89       200
   macro avg       0.81      0.8